# 05 — Modèles de base, comparaison des couches et ablation

**Objectif** : mesurer ce que chaque couche de caractéristiques (URL, contenu, externe) apporte à la détection du phishing, avec les 4 classifieurs du cours (KNN, arbre de décision, Random Forest, SVM).

**Règles suivies dans tout le notebook :**
- On travaille **uniquement sur `train.csv`** (9 003 URL). Le test verrouillé (`test_LOCKED.csv`) n'est pas ouvert.
- Validation croisée **groupée par site** : `GroupKFold(5)` sur la colonne `group` (définie dans `04_split`), pour qu'un même site ne soit jamais à la fois en entraînement et en validation.
- Classe positive : `y = 1` = **phishing**. Critère principal : **F1** (pas l'accuracy).
- `random_state=42` partout.

In [3]:
import pandas as pd

In [4]:
train = pd.read_csv("../data/processed/train.csv")

## 1. Les 3 couches de caractéristiques

Les 87 colonnes de Hannousse sont réparties en 3 couches, délimitées par le nom de leur première et dernière colonne :

| Couche | Colonnes | Nombre | Ce qu'elle mesure | Coût en production |
| --- | --- | --- | --- | --- |
| **URL** | `length_url` → `statistical_report` | 56 | forme de l'URL (longueur, caractères spéciaux, sous-domaines, mots suspects…) | gratuit, instantané |
| **Contenu** | `nb_hyperlinks` → `domain_with_copyright` | 24 | structure de la page HTML (liens, formulaires, iframes…) | télécharger la page |
| **Externe** | `whois_registered_domain` → `page_rank` | 7 | WHOIS, index Google, trafic, PageRank | services tiers (lents, parfois disparus) |

`url`, `status` et les colonnes de groupe ne sont **pas** des caractéristiques : ce sont des identifiants.

In [5]:
cols = list(train.columns)
URL_COLS = cols[cols.index("length_url") : cols.index("statistical_report") + 1]
CONTENT_COLS = cols[cols.index("nb_hyperlinks") : cols.index("domain_with_copyright") + 1]
EXTERNAL_COLS = cols[cols.index("whois_registered_domain") : cols.index("page_rank") + 1]

print(len(URL_COLS), len(CONTENT_COLS), len(EXTERNAL_COLS))

56 24 7


In [6]:
X = train[URL_COLS + CONTENT_COLS + EXTERNAL_COLS]
y = (train["status"] == "phishing").astype(int)   # 1 = phishing
groups = train["group"]

print(X.shape, y.mean().round(3), groups.nunique())

(9003, 87) 0.492 5948


## 2. Premier modèle : Random Forest sur la couche URL

Random Forest avec ses réglages par défaut, évalué par validation croisée groupée (5 plis). On obtient un score par pli : la **moyenne** donne la performance attendue, l'**écart-type** sa stabilité.

In [7]:
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier


In [8]:
cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

In [9]:
scores = cross_validate(rf, X[URL_COLS], y, groups=groups, cv=cv,scoring=["f1", "precision", "recall", "roc_auc"])

In [10]:
pd.DataFrame(scores).drop(columns=["fit_time", "score_time"]).agg(["mean", "std"]).round(4)

,test_f1,test_precision,test_recall,test_roc_auc
mean,0.8784,0.8729,0.8843,0.9521
std,0.0246,0.0283,0.0250,0.0125


**Résultat** : F1 = **0,878** (± 0,025), ROC-AUC = 0,952. La seule forme de l'URL, sans page ni popularité, apporte déjà un vrai signal. L'écart-type d'environ 2,5 points signifie qu'une différence de moins de ~2 points entre deux modèles n'est pas forcément réelle.

## 3. Pourquoi découper par site : `GroupKFold` contre `KFold` aléatoire

Même modèle, mêmes données, seul le découpage change. Avec `KFold` (ce que font la plupart des publications), les URL d'un même site sont dispersées entre entraînement et validation : le modèle peut « reconnaître » un site déjà vu.

In [11]:
from sklearn.model_selection import KFold

In [12]:
cv_random = KFold(n_splits=5, shuffle=True, random_state=42)

In [13]:
scores_random = cross_validate(rf, X[URL_COLS], y, cv=cv_random,scoring=["f1", "precision", "recall", "roc_auc"])

In [14]:
comparison = pd.DataFrame({
    "par site (GroupKFold)": pd.DataFrame(scores).mean(),
    "aléatoire (KFold)": pd.DataFrame(scores_random).mean(),
}).drop(["fit_time", "score_time"]).round(4)
comparison["écart"] = comparison["aléatoire (KFold)"] - comparison["par site (GroupKFold)"]
comparison

,par site (GroupKFold),aléatoire (KFold),écart
test_f1,0.8784,0.9082,0.0298
test_precision,0.8729,0.9112,0.0383
test_recall,0.8843,0.9053,0.0210
test_roc_auc,0.9521,0.9675,0.0154


**Résultat** : le découpage aléatoire gonfle le F1 de **+3,0 points** (0,878 → 0,908) et la précision de +3,8 points, avec exactement le même modèle. Ce gain vient de la mémorisation des sites, pas de la détection. C'est plus que ce que le stacking apporte en général (1–2 points) : **le choix du découpage pèse plus que le choix du modèle.** Tous les résultats suivants utilisent `GroupKFold`.

## 4. Les 4 classifieurs sur la couche URL

Réglages par défaut. `StandardScaler` pour KNN et SVM (ils calculent des distances), pas pour l'arbre et le RF (ils comparent chaque colonne à un seuil). La normalisation est dans un `Pipeline` : elle est apprise sur les plis d'entraînement seulement.

In [15]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [16]:
models = {
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "Arbre": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    "SVM": make_pipeline(StandardScaler(), SVC(random_state=42)),
}

In [17]:
rows = {}
for name, model in models.items():
    s = cross_validate(model, X[URL_COLS], y, groups=groups, cv=cv,
                       scoring=["f1", "precision", "recall", "roc_auc"])
    s = pd.DataFrame(s)
    rows[name] = {"F1": s["test_f1"].mean(), "F1 std": s["test_f1"].std(),
                  "précision": s["test_precision"].mean(), "rappel": s["test_recall"].mean(),
                  "ROC-AUC": s["test_roc_auc"].mean()}

In [18]:
pd.DataFrame(rows).T.round(4).sort_values("F1", ascending=False)

,F1,F1 std,précision,rappel,ROC-AUC
RF,0.8784,0.0246,0.8729,0.8843,0.9521
SVM,0.8683,0.0314,0.8751,0.8619,0.9410
KNN,0.8367,0.0524,0.8723,0.8050,0.9038
Arbre,0.8208,0.0465,0.8300,0.8129,0.8268


**Résultat** (F1) : RF **0,878** > SVM 0,868 > KNN 0,837 > Arbre 0,821.
- RF en tête, comme dans l'état de l'art, mais SVM est à 1 point, en dessous de l'écart-type : pas de différence significative.
- L'arbre seul sur-apprend (pas de limite de profondeur, ROC-AUC 0,827) ; le RF (300 arbres moyennés) gagne 6 points.

## 5. Les 4 classifieurs × les 3 couches (12 modèles)

Avant d'utiliser la couche externe, on traite les codes « inconnu » (décision prise dans `03_mendeley_eda`) :
- `domain_age` et `domain_registration_length` < 0 (-1, -2, -12 = échec WHOIS) → `NaN` ;
- dans le `Pipeline` : `SimpleImputer(strategy="median", add_indicator=True)` remplace par la médiane **des plis d'entraînement** et ajoute une colonne « était inconnu ».

In [19]:
from sklearn.impute import SimpleImputer

In [20]:
# -1, -2, -12 = « inconnu » (échec WHOIS), pas un vrai nombre -> valeur manquante
X = X.copy()
for col in ["domain_age", "domain_registration_length"]:
    X[col] = X[col].mask(X[col] < 0)


In [21]:
def make_models():
    impute = lambda: SimpleImputer(strategy="median", add_indicator=True)
    return {
        "KNN": make_pipeline(impute(), StandardScaler(), KNeighborsClassifier()),
        "Arbre": make_pipeline(impute(), DecisionTreeClassifier(random_state=42)),
        "RF": make_pipeline(impute(), RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)),
        "SVM": make_pipeline(impute(), StandardScaler(), SVC(random_state=42)),
    }

In [22]:
LAYERS = {"URL": URL_COLS, "Contenu": CONTENT_COLS, "Externe": EXTERNAL_COLS}

In [23]:
results = []
for layer, layer_cols in LAYERS.items():
    for name, model in make_models().items():
        s = pd.DataFrame(cross_validate(model, X[layer_cols], y, groups=groups, cv=cv,
                                        scoring=["f1", "roc_auc"]))
        results.append({"couche": layer, "modèle": name, "F1": s["test_f1"].mean(),
                        "F1 std": s["test_f1"].std(), "ROC-AUC": s["test_roc_auc"].mean()})

results = pd.DataFrame(results)
results.pivot(index="modèle", columns="couche", values="F1").round(4)

couche,Contenu,Externe,URL
modèle,,,
Arbre,0.7616,0.8616,0.8208
KNN,0.7973,0.8800,0.8368
RF,0.8006,0.8992,0.8784
SVM,0.8203,0.8738,0.8683


**Résultat** (F1) :

| Modèle | URL | Contenu | Externe |
| --- | --- | --- | --- |
| KNN | 0,837 | 0,797 | 0,880 |
| Arbre | 0,821 | 0,762 | 0,862 |
| RF | **0,878** | 0,801 | **0,899** |
| SVM | 0,868 | **0,820** | 0,874 |

- **Les 7 caractéristiques externes battent les 56 de l'URL.** Elles mesurent la **popularité** du site (la règle `google_index` seule donnait déjà 0,869), pas le phishing.
- Le contenu seul est la couche la plus faible.
- Le meilleur algorithme change selon la couche (RF, SVM) : argument pour le **stacking** (meilleur modèle par couche, puis combinaison).

## 6. Ablation : URL → + Contenu → + Externe

On ajoute les couches une à une, dans l'ordre de leur coût en production, pour mesurer ce que **chacune ajoute** aux précédentes.

In [24]:
ABLATION = {
    "1. URL": URL_COLS,
    "2. URL + Contenu": URL_COLS + CONTENT_COLS,
    "3. URL + Contenu + Externe": URL_COLS + CONTENT_COLS + EXTERNAL_COLS,
}

abl = []
for step, step_cols in ABLATION.items():
    for name, model in make_models().items():
        s = pd.DataFrame(cross_validate(model, X[step_cols], y, groups=groups, cv=cv,
                                        scoring=["f1", "roc_auc"]))
        abl.append({"étape": step, "modèle": name, "F1": s["test_f1"].mean(),
                    "F1 std": s["test_f1"].std(), "ROC-AUC": s["test_roc_auc"].mean()})

abl = pd.DataFrame(abl)
abl.pivot(index="modèle", columns="étape", values="F1").round(4)


étape,1. URL,2. URL + Contenu,3. URL + Contenu + Externe
modèle,,,
Arbre,0.8208,0.8276,0.9101
KNN,0.8368,0.8528,0.8966
RF,0.8784,0.9168,0.9546
SVM,0.8683,0.9054,0.9392


**Résultat** (F1) :

| Modèle | URL | + Contenu | + Externe |
| --- | --- | --- | --- |
| KNN | 0,837 | 0,853 | 0,897 |
| Arbre | 0,821 | 0,828 | 0,910 |
| RF | 0,878 | **0,917** | **0,955** |
| SVM | 0,868 | 0,905 | 0,939 |

- **Le contenu apporte +3,8 points au RF** : seul il est faible, mais il complète l'URL (l'URL dit *où* est la page, le contenu *ce qu'elle fait*).
- **L'externe ajoute encore +3,8 points, mais grâce à la popularité** : l'arbre seul gagne +8,3 points en s'appuyant sur `google_index`.
- **Chiffre de référence retenu : RF, URL + Contenu, F1 = 0,917.** Le 0,955 n'est pas transposable (popularité, services disparus, biais de sélection des légitimes).

## 7. Réglage des hyperparamètres (GridSearch)

`GridSearchCV` essaie toutes les combinaisons de la grille, chacune évaluée avec **les mêmes plis `GroupKFold`** (`groups=groups` dans `fit`). Le choix se fait sur le train uniquement.

- **RF** sur la configuration de référence (URL + Contenu) : `max_depth`, `min_samples_leaf`, `max_features` (12 combinaisons), qui contrôlent le sur-apprentissage.
- **SVM** sur la couche contenu (où il est le meilleur) : `C` (tolérance aux erreurs) et `gamma` (portée de chaque exemple), 12 combinaisons espacées d'un facteur 10.

In [25]:
from sklearn.model_selection import GridSearchCV

In [26]:
HONEST_COLS = URL_COLS + CONTENT_COLS
param_grid = {
    "randomforestclassifier__max_depth": [None, 20],
    "randomforestclassifier__min_samples_leaf": [1, 2, 5],
    "randomforestclassifier__max_features": ["sqrt", 0.3],
}

In [27]:
grid = GridSearchCV(make_models()["RF"], param_grid, cv=cv, scoring="f1", n_jobs=1)
grid.fit(X[HONEST_COLS], y, groups=groups)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'randomforestclassifier__max_depth': [None, 20], 'randomforestclassifier__max_features': ['sqrt', 0.3], 'randomforestclassifier__min_samples_leaf': [1, 2, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",GroupKFold(n_... shuffle=True)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score..

In [28]:
print("meilleurs paramètres :", grid.best_params_)
print("F1 :", round(grid.best_score_, 4),
      "| std :", round(grid.cv_results_["std_test_score"][grid.best_index_], 4))

meilleurs paramètres : {'randomforestclassifier__max_depth': None, 'randomforestclassifier__max_features': 'sqrt', 'randomforestclassifier__min_samples_leaf': 1}
F1 : 0.9168 | std : 0.0184


**Résultat** : le GridSearch choisit exactement les **réglages par défaut** (`max_depth=None`, `max_features="sqrt"`, `min_samples_leaf=1`), F1 = **0,917** (± 0,018). Le RF est robuste à ses hyperparamètres : le réglage n'apporte rien.

In [29]:
param_grid_svm = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__gamma": ["scale", 0.01, 0.1],
}

grid_svm = GridSearchCV(make_models()["SVM"], param_grid_svm, cv=cv, scoring="f1", n_jobs=-1)
grid_svm.fit(X[CONTENT_COLS], y, groups=groups)

print("meilleurs paramètres :", grid_svm.best_params_)
print("F1 :", round(grid_svm.best_score_, 4),
      "| std :", round(grid_svm.cv_results_["std_test_score"][grid_svm.best_index_], 4))


meilleurs paramètres : {'svc__C': 10, 'svc__gamma': 'scale'}
F1 : 0.8351 | std : 0.0335


**Résultat** : meilleur réglage `C=10`, `gamma="scale"`, F1 = **0,835** (± 0,034) contre 0,820 par défaut.

| `C` \ `gamma` | 0,01 | 0,1 | `scale` |
| --- | --- | --- | --- |
| 0,1 | 0,782 | 0,797 | 0,803 |
| 1 (défaut) | 0,810 | 0,829 | 0,820 |
| **10** | 0,816 | 0,829 | **0,835** |
| 100 | 0,825 | 0,815 | 0,827 |

- Forme en cloche : `C` trop petit → sous-apprentissage, `C` trop grand → sur-apprentissage (compromis biais/variance).
- Gain de +1,5 point, inférieur à l'écart-type : on garde `C=10`, sans conclure à un gain significatif.

## 8. Stacking : un modèle par couche + méta-modèle

- **Modèles de base spécialisés** : RF sur la couche URL, SVM (`C=10`) sur la couche contenu. `ColumnTransformer` donne à chacun **seulement sa couche**.
- **Méta-modèle** : arbre de décision de profondeur 3 (lisible), entraîné sur les prédictions hors-pli des deux modèles de base.
- **Piège évité** : `StackingClassifier` ne transmet pas `groups` à sa validation croisée interne. On écrit donc la boucle extérieure à la main, et on lui passe des **plis internes déjà groupés par site** (`inner_cv`). Sinon, le méta-modèle apprendrait sur des prédictions trop optimistes (même fuite qu'en partie 3).
- **Comparaison** : avec la fusion précoce de l'ablation (un seul RF sur URL + Contenu, F1 = 0,917), sur les mêmes plis extérieurs.

In [30]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import f1_score, roc_auc_score

In [31]:
def select(columns):
    return ColumnTransformer([("select", "passthrough", columns)])

def make_stack(inner_cv):
    url_rf = make_pipeline(select(URL_COLS),
                           RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))
    content_svm = make_pipeline(select(CONTENT_COLS),
                                SimpleImputer(strategy="median", add_indicator=True),
                                StandardScaler(), SVC(C=10, random_state=42))
    return StackingClassifier(
        estimators=[("url_rf", url_rf), ("contenu_svm", content_svm)],
        final_estimator=DecisionTreeClassifier(max_depth=3, random_state=42),
        cv=inner_cv,
    )

Xh, f1s, aucs = X[HONEST_COLS], [], []
for train_idx, val_idx in cv.split(Xh, y, groups):
    X_tr, y_tr, g_tr = Xh.iloc[train_idx], y.iloc[train_idx], groups.iloc[train_idx]
    inner_cv = list(GroupKFold(n_splits=5, shuffle=True, random_state=42).split(X_tr, y_tr, g_tr))

    stack = make_stack(inner_cv).fit(X_tr, y_tr)
    f1s.append(f1_score(y.iloc[val_idx], stack.predict(Xh.iloc[val_idx])))
    aucs.append(roc_auc_score(y.iloc[val_idx], stack.predict_proba(Xh.iloc[val_idx])[:, 1]))

print("Stacking  F1 :", round(np.mean(f1s), 4), "| std :", round(np.std(f1s, ddof=1), 4),
      "| ROC-AUC :", round(np.mean(aucs), 4))
print("Fusion précoce (RF, URL + Contenu) F1 : 0.9168")

Stacking  F1 : 0.9126 | std : 0.0251 | ROC-AUC : 0.9578
Fusion précoce (RF, URL + Contenu) F1 : 0.9168


**Résultat** :

| Approche | F1 | Écart-type | ROC-AUC |
| --- | --- | --- | --- |
| **Fusion précoce** (un RF sur URL + Contenu) | **0,917** | 0,018 | — |
| Stacking (RF URL + SVM Contenu → arbre de profondeur 3) | 0,913 | 0,025 | 0,958 |

- **Pas de gain** : −0,4 point, dans le bruit (égalité statistique), et un peu moins stable.
- Le RF fusionne déjà les couches lui-même (un même arbre combine caractéristiques URL et contenu) ; le méta-modèle ne voit que 2 avis résumés.
- Avec une régression logistique comme méta-modèle (test complémentaire) : F1 0,916, ROC-AUC 0,973 → même conclusion.
- Cohérent avec l'état de l'art (stacking : +1–2 points au mieux).

## Conclusion du notebook

| Étape | Résultat clé |
| --- | --- |
| Découpage | `KFold` aléatoire gonfle le F1 de **+3 points** par mémorisation des sites → tout est évalué avec `GroupKFold`. |
| Couches seules | URL 0,878 · Contenu 0,820 · Externe 0,899 (popularité, pas phishing). |
| Ablation (RF) | URL **0,878** → + Contenu **0,917** → + Externe 0,955. |
| GridSearch | RF : réglages par défaut optimaux. SVM : +1,5 point (non significatif). |
| Stacking | 0,913, pas mieux que la fusion précoce. |

**Modèle retenu** : Random Forest (réglages par défaut, 300 arbres) sur **URL + Contenu**, F1 = **0,917 ± 0,018** en validation croisée groupée. C'est le meilleur, le plus stable et le plus simple (rasoir d'Ockham), et il est compatible avec SHAP pour les explications.

Le test verrouillé n'a pas été utilisé.